# Autoresearch run analysis

This notebook analyzes the append-only `runs.csv` log. It is exploratory and observational: using the BO tool is optional, baseline/treatment labels are manually assigned, and any correlation shown here is **not** causal evidence or a significance test. `results.tsv` is intentionally not used.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.25})

RUNS_PATH = Path("runs.csv")
MIN_SEARCH_SPACE_OBSERVATIONS = 4

# Fill this in after each night. Branches omitted here stay in diagnostic views
# but are excluded from baseline/treatment comparisons.
BRANCH_CONDITIONS = {
    # "autoresearch/mar5": "baseline",
    # "autoresearch/mar12": "treatment",
}

SOURCE_COLORS = {"agent": "#1f77b4", "tool": "#ff7f0e"}
SOURCE_MARKERS = {"agent": "o", "tool": "^"}


## Load and clean the run log

Successful observations have a numeric `val_bpb`; failed or incomplete attempts remain available for attempt counts and BO diagnostics. Legacy rows without `branch` are retained with a blank raw branch and shown as `(unknown)`.

In [ ]:
CANONICAL_COLUMNS = [
    "timestamp", "iteration", "fingerprint", "lr", "weight_decay", "val_bpb",
    "wall_clock_seconds", "source", "requested_lr", "requested_weight_decay",
    "n_prior_obs", "cold_start", "branch",
]
NUMERIC_COLUMNS = [
    "iteration", "lr", "weight_decay", "val_bpb", "wall_clock_seconds",
    "requested_lr", "requested_weight_decay", "n_prior_obs",
]

if RUNS_PATH.exists() and RUNS_PATH.stat().st_size:
    runs = pd.read_csv(RUNS_PATH)
else:
    runs = pd.DataFrame(columns=CANONICAL_COLUMNS)
    print(f"No runs yet: {RUNS_PATH} is missing or empty. Charts will explain when they have no data.")

for column in CANONICAL_COLUMNS:
    if column not in runs:
        runs[column] = pd.NA

runs["timestamp"] = pd.to_datetime(runs["timestamp"], errors="coerce", utc=True)
for column in NUMERIC_COLUMNS:
    runs[column] = pd.to_numeric(runs[column], errors="coerce")

cold_start_values = runs["cold_start"].astype("string").str.strip().str.lower()
runs["cold_start"] = cold_start_values.map({"true": True, "1": True, "yes": True, "false": False, "0": False, "no": False}).astype("boolean")
runs["source"] = runs["source"].fillna("unknown").astype("string").str.strip().replace("", "unknown")
runs["fingerprint"] = runs["fingerprint"].fillna("(missing)").astype("string").str.strip().replace("", "(missing)")
runs["branch"] = runs["branch"].fillna("").astype("string").str.strip()
runs["branch_display"] = runs["branch"].replace("", "(unknown)")
runs["condition"] = runs["branch"].map(BRANCH_CONDITIONS).fillna("unlabeled")
successful_runs = runs[runs["val_bpb"].notna()].copy()
incomplete_runs = runs[runs["val_bpb"].isna()].copy()

print(f"All attempts: {len(runs)} | successful: {len(successful_runs)} | incomplete/failed: {len(incomplete_runs)}")
print(f"Columns available: {', '.join(runs.columns)}")
display(runs.head(10))


## Per-branch summary

Tool-usage fraction uses every attempt, including incomplete tool attempts. Best `val_bpb` uses successful observations only; lower is better.

In [ ]:
if runs.empty:
    branch_summary = pd.DataFrame(columns=["branch", "condition", "n_agent_runs", "n_tool_runs", "tool_usage_fraction", "best_val_bpb", "n_distinct_fingerprints"])
else:
    branch_summary = (
        runs.groupby(["branch_display", "condition"], dropna=False)
        .agg(
            n_agent_runs=("source", lambda values: (values == "agent").sum()),
            n_tool_runs=("source", lambda values: (values == "tool").sum()),
            n_total_runs=("source", "size"),
            n_distinct_fingerprints=("fingerprint", "nunique"),
        )
        .reset_index()
        .rename(columns={"branch_display": "branch"})
    )
    best_by_branch = successful_runs.groupby("branch_display")["val_bpb"].min()
    branch_summary["best_val_bpb"] = branch_summary["branch"].map(best_by_branch)
    branch_summary["tool_usage_fraction"] = branch_summary["n_tool_runs"] / branch_summary["n_total_runs"]
    branch_summary = branch_summary.drop(columns="n_total_runs").sort_values(["condition", "branch"])

display(branch_summary)


## Best validation BPB over time

Raw points are colored by source. The black line is the branch-wide running minimum, ordered by iteration when present and otherwise by timestamp. Incomplete attempts are counted above but have no validation metric to plot.

In [ ]:
if successful_runs.empty:
    print("No successful runs available for a trajectory.")
else:
    branches = sorted(successful_runs["branch_display"].unique())
    fig, axes = plt.subplots(len(branches), 1, figsize=(11, 4 * len(branches)), sharey=True, squeeze=False)
    for axis, branch in zip(axes.ravel(), branches):
        data = successful_runs[successful_runs["branch_display"] == branch].copy()
        if data["iteration"].notna().any():
            data = data.sort_values(["iteration", "timestamp"], na_position="last")
            x = data["iteration"]
            x_label = "Iteration"
        else:
            data = data.sort_values("timestamp")
            x = data["timestamp"]
            x_label = "Timestamp"
        for source, source_data in data.groupby("source"):
            axis.scatter(x.loc[source_data.index], source_data["val_bpb"], label=source, color=SOURCE_COLORS.get(source, "#777777"), alpha=0.8)
        axis.step(x, data["val_bpb"].cummin(), where="mid", color="black", linewidth=2, label="running best")
        axis.set_title(f'{branch} ({data["condition"].iloc[0]})')
        axis.set_xlabel(x_label)
        axis.set_ylabel("Validation BPB (lower is better)")
        axis.legend()
    fig.tight_layout()


## Tool usage versus outcome

Only manually labeled baseline/treatment branches appear below. This is a branch-level observational view, not a formal test; with fewer than three eligible branches it is especially descriptive rather than informative.

In [ ]:
labeled_branches = branch_summary[branch_summary["condition"].isin(["baseline", "treatment"]) & branch_summary["best_val_bpb"].notna()].copy()
print(f"Eligible labeled branches: {len(labeled_branches)}")
if len(labeled_branches) < 3:
    print("Caution: fewer than three labeled branches; do not interpret this as a meaningful correlation.")
if labeled_branches.empty:
    print("No labeled branches with successful observations yet.")
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = labeled_branches["condition"].map({"baseline": "#1f77b4", "treatment": "#ff7f0e"})
    ax.scatter(labeled_branches["tool_usage_fraction"], labeled_branches["best_val_bpb"], c=colors, s=85)
    for _, point in labeled_branches.iterrows():
        ax.annotate(point["branch"], (point["tool_usage_fraction"], point["best_val_bpb"]), xytext=(5, 5), textcoords="offset points", fontsize=8)
    ax.set(xlabel="Tool-usage fraction (all attempts)", ylabel="Best validation BPB (lower is better)", title="Labeled branches: tool usage vs. best outcome")
    plt.show()


## Fingerprint-level view

In [ ]:
if runs.empty:
    fingerprint_summary = pd.DataFrame(columns=["fingerprint", "observation_count", "cold_start_count", "branches_touched", "best_val_bpb"])
else:
    fingerprint_summary = (
        runs.groupby("fingerprint", dropna=False)
        .agg(
            observation_count=("fingerprint", "size"),
            cold_start_count=("cold_start", lambda values: (values == True).sum()),
            branches_touched=("branch_display", lambda values: ", ".join(sorted(values.dropna().unique()))),
        )
        .reset_index()
    )
    fingerprint_summary["best_val_bpb"] = fingerprint_summary["fingerprint"].map(successful_runs.groupby("fingerprint")["val_bpb"].min())
    fingerprint_summary = fingerprint_summary.sort_values(["observation_count", "best_val_bpb"], ascending=[False, True])
display(fingerprint_summary)


## BO tool diagnostics

Requested-versus-actual plots check the protected CLI contract. Cold-start and prior-observation plots describe how the tool was used, not whether it caused an outcome.

In [ ]:
tool_runs = runs[runs["source"] == "tool"].copy()
tool_success = successful_runs[successful_runs["source"] == "tool"].copy()
contract_rows = tool_success.dropna(subset=["requested_lr", "requested_weight_decay", "lr", "weight_decay"])
print(f"Tool attempts: {len(tool_runs)} | successful tool runs: {len(tool_success)} | requested/actual pairs: {len(contract_rows)}")

if contract_rows.empty:
    print("No complete requested-versus-actual tool pairs yet.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for axis, requested, actual, label in [
        (axes[0], "requested_lr", "lr", "Learning rate"),
        (axes[1], "requested_weight_decay", "weight_decay", "Weight decay"),
    ]:
        axis.scatter(contract_rows[requested], contract_rows[actual], alpha=0.8)
        lower = min(contract_rows[requested].min(), contract_rows[actual].min())
        upper = max(contract_rows[requested].max(), contract_rows[actual].max())
        axis.plot([lower, upper], [lower, upper], "k--", label="y = x")
        axis.set(xlabel=f"Requested {label}", ylabel=f"Actual {label}", title=f"Requested vs. actual {label}")
        axis.legend()
    fig.tight_layout()

if tool_runs.empty:
    print("No tool calls yet for cold-start or prior-observation diagnostics.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    timeline_x = tool_runs["iteration"] if tool_runs["iteration"].notna().any() else tool_runs["timestamp"]
    axes[0].scatter(timeline_x, tool_runs["cold_start"].astype(float), c=tool_runs["cold_start"].map({True: "#ff7f0e", False: "#1f77b4"}).fillna("#777777"))
    axes[0].set(xlabel="Iteration or timestamp", ylabel="Cold start (1=yes)", title="Cold starts over time", yticks=[0, 1])
    cold_counts = tool_runs.assign(cold_start_label=tool_runs["cold_start"].map({True: "cold", False: "warm"}).fillna("unknown")).groupby(["fingerprint", "cold_start_label"]).size().unstack(fill_value=0)
    cold_counts.plot(kind="bar", stacked=True, ax=axes[1], color={"cold": "#ff7f0e", "warm": "#1f77b4", "unknown": "#777777"})
    axes[1].set(xlabel="Fingerprint", ylabel="Tool calls", title="Cold/warm calls by fingerprint")
    axes[1].tick_params(axis="x", rotation=30)
    axes[2].scatter(timeline_x, tool_runs["n_prior_obs"], c=tool_runs["cold_start"].map({True: "#ff7f0e", False: "#1f77b4"}).fillna("#777777"))
    axes[2].set(xlabel="Iteration or timestamp", ylabel="n_prior_obs", title="Prior observations at tool calls")
    fig.tight_layout()


## LR / weight-decay search space

Each eligible fingerprint is plotted separately. Color encodes validation BPB (lower is better); marker shape distinguishes hand-picked and tool-proposed evaluations.

In [ ]:
searchable = successful_runs.dropna(subset=["lr", "weight_decay", "val_bpb"]).copy()
eligible_fingerprints = searchable.groupby("fingerprint").size()
eligible_fingerprints = eligible_fingerprints[eligible_fingerprints >= MIN_SEARCH_SPACE_OBSERVATIONS].index.tolist()
if not eligible_fingerprints:
    print(f"No fingerprint has at least {MIN_SEARCH_SPACE_OBSERVATIONS} successful hyperparameter observations yet.")
else:
    plot_data = searchable[searchable["fingerprint"].isin(eligible_fingerprints)]
    norm = plt.Normalize(plot_data["val_bpb"].min(), plot_data["val_bpb"].max() if plot_data["val_bpb"].max() > plot_data["val_bpb"].min() else plot_data["val_bpb"].min() + 1e-12)
    fig, axes = plt.subplots(len(eligible_fingerprints), 1, figsize=(9, 5 * len(eligible_fingerprints)), squeeze=False)
    for axis, fingerprint in zip(axes.ravel(), eligible_fingerprints):
        data = plot_data[plot_data["fingerprint"] == fingerprint]
        for source, source_data in data.groupby("source"):
            scatter = axis.scatter(source_data["lr"], source_data["weight_decay"], c=source_data["val_bpb"], cmap="viridis_r", norm=norm, marker=SOURCE_MARKERS.get(source, "s"), s=70, edgecolors="black", linewidths=0.4, label=source)
        axis.set(xlabel="Learning rate", ylabel="Weight decay", title=f"Search space: {fingerprint}")
        axis.legend(title="Source")
    fig.colorbar(scatter, ax=axes.ravel().tolist(), label="Validation BPB (lower is better)")
    fig.tight_layout()


## Agent versus tool, matched by fingerprint

This compares only fingerprints with successful points from both sources. It remains observational: time, agent decisions, and small sample sizes can all confound apparent differences.

In [ ]:
source_counts = successful_runs.groupby(["fingerprint", "source"]).size().unstack(fill_value=0)
matched_fingerprints = source_counts.index[(source_counts.get("agent", 0) > 0) & (source_counts.get("tool", 0) > 0)].tolist()
if not matched_fingerprints:
    print("No fingerprint has successful observations from both agent and tool yet.")
else:
    fig, axes = plt.subplots(len(matched_fingerprints), 1, figsize=(9, 4 * len(matched_fingerprints)), sharey=True, squeeze=False)
    rng = np.random.default_rng(0)
    for axis, fingerprint in zip(axes.ravel(), matched_fingerprints):
        data = successful_runs[successful_runs["fingerprint"] == fingerprint]
        agent_values = data.loc[data["source"] == "agent", "val_bpb"].dropna().to_numpy()
        tool_values = data.loc[data["source"] == "tool", "val_bpb"].dropna().to_numpy()
        axis.boxplot([agent_values, tool_values], labels=["agent", "tool"], showfliers=False)
        axis.scatter(1 + rng.uniform(-0.08, 0.08, len(agent_values)), agent_values, color=SOURCE_COLORS["agent"], alpha=0.8)
        axis.scatter(2 + rng.uniform(-0.08, 0.08, len(tool_values)), tool_values, color=SOURCE_COLORS["tool"], alpha=0.8)
        axis.set(title=f"{fingerprint}", ylabel="Validation BPB (lower is better)")
    fig.tight_layout()
